In [ ]:
#INITIAL GATHERING OF DATA GNOMAD_AF, SIFT, CADD, Is_Nonsense

import pandas as pd
import requests
import time
import os

print("--- Phase 1: Loading and Cleaning Data ---")
# 1. Load the clean ClinVar data
df = pd.read_csv('clinvar_ml_labels.csv', low_memory=False)

# 2. Drop missing rows and unmapped -1 positions
df = df.dropna(subset=['ReferenceAlleleVCF', 'AlternateAlleleVCF', 'PositionVCF'])
df = df[df['PositionVCF'] != -1]

# 3. Keep ONLY simple 1-letter mutations (SNPs) to prevent API 400 errors
df = df[
    (df['ReferenceAlleleVCF'].str.len() == 1) &
    (df['AlternateAlleleVCF'].str.len() == 1)
]

print("--- Phase 2: Balancing the Dataset ---")
# 4. Grab exactly 6,000 of each to make the AI learn properly (12k total)
# Using a random_state ensures you get the exact same random mix if you run it again
pathogenic_df = df[df['ClinicalSignificance'] == 'Pathogenic'].sample(n=6000, random_state=42)
benign_df = df[df['ClinicalSignificance'] == 'Benign'].sample(n=6000, random_state=42)

# Combine and shuffle the rows so they aren't all Pathogenic at the top
training_df = pd.concat([pathogenic_df, benign_df]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Dataset balanced! Total rows to process: {len(training_df)}")

# 5. Define the API fetching function
def get_variant_clues(chrom, pos, alt):
    """Fetches gnomAD AF, SIFT, CADD (replaces PhyloP), and Nonsense flag."""
    server = "https://rest.ensembl.org"

    # THE UPGRADE: We dropped the heavy dbNSFP and added native CADD=1
    endpoint = f"/vep/human/region/{chrom}:{pos}-{pos}/{alt}?af=1&CADD=1"

    try:
        response = requests.get(server + endpoint, headers={"Content-Type": "application/json"})
        if not response.ok:
            return 0.0, 0.05, 0.0, 0

        data = response.json()[0]

        # --- 1. Extract gnomAD Allele Frequency ---
        gnomad_af = 0.0
        for colocated in data.get('colocated_variants', []):
            freqs = colocated.get('frequencies', {}).get(alt, {})
            # VEP sometimes keys it differently depending on the specific genome build
            for key in ['gnomad', 'gnomade', 'gnomadg']:
                if key in freqs:
                    gnomad_af = float(freqs[key])
                    break
            if gnomad_af > 0.0:
                break

        # --- 2. Extract Transcript Features (SIFT, CADD, Nonsense) ---
        sift_score = 1.0
        cadd_score = 0.0   # Replaces PhyloP (Scale of 0 to ~40)
        is_nonsense = 0
        sift_found = False

        for t in data.get('transcript_consequences', []):

            # Nonsense check
            if 'stop_gained' in t.get('consequence_terms', []):
                is_nonsense = 1

            # SIFT check (Find the lowest/most damaging score)
            if 'sift_score' in t:
                sift_found = True
                if t['sift_score'] < sift_score:
                    sift_score = t['sift_score']

            # CADD check (Find the highest/most damaging score)
            if 'cadd_phred' in t:
                if t['cadd_phred'] > cadd_score:
                    cadd_score = t['cadd_phred']

        # If no SIFT was found, default to a neutral number
        if not sift_found:
            sift_score = 0.05

        return gnomad_af, sift_score, cadd_score, is_nonsense

    except Exception:
        return 0.0, 0.05, 0.0, 0

print("--- Phase 3: Fetching API Data (Expected time: ~45 mins) ---")

output_file = 'final_training_data.csv'

# Create the CSV with headers if it doesn't exist yet
if not os.path.isfile(output_file):
    with open(output_file, 'w') as f:
        f.write("Variant_ID,gnomAD_AF,SIFT_Score,PhyloP_Score,Is_Nonsense,Is_Pathogenic\n")

# Process the rows
for index, row in training_df.iterrows():
    chrom = str(row['Chromosome']).replace('chr', '')
    pos = int(row['PositionVCF'])
    ref = row['ReferenceAlleleVCF']
    alt = row['AlternateAlleleVCF']

    # Binary Target: 1 for Pathogenic, 0 for Benign
    target = 1 if row['ClinicalSignificance'] == 'Pathogenic' else 0

    # Fetch all 4 clues
    af, sift, phylop, nonsense = get_variant_clues(chrom, pos, alt)

    # Format the data line
    variant_id = f"{chrom}:{pos}:{ref}:{alt}"
    csv_line = f"{variant_id},{af},{sift},{phylop},{nonsense},{target}\n"

    # Append the single row directly to the CSV file
    with open(output_file, 'a') as f:
        f.write(csv_line)

    # THROTTLE: 0.1 second pause (Keeps you safely under the 15 requests/sec limit)
    time.sleep(0.1)

    # Console output to track progress
    if (index + 1) % 100 == 0:
        print(f"✅ Processed {index + 1} / {len(training_df)} rows...")

print("🎉 DONE! Your dataset is fully extracted and saved as final_training_data.csv")

In [ ]:
!pip install tqdm
import pandas as pd
import requests
import time
from tqdm import tqdm

print("1. Loading existing dataset...")
df = pd.read_csv('final_training_data.csv')

# Testing with 50 rows to verify!
df_test = df.head(12000).copy()

is_missense_list = []
is_synonymous_list = [] # The new "Safe" feature
is_splice_list = []

print(f"2. Fetching structural features for {len(df_test)} variants...\n")
server = "https://rest.ensembl.org"

for index, row in tqdm(df_test.iterrows(), total=len(df_test), desc="Pinging Ensembl"):
    variant_id = row['Variant_ID']
    parts = variant_id.split(':')
    chrom, pos, ref, alt = parts[0], parts[1], parts[2], parts[3]

    endpoint = f"/vep/human/region/{chrom}:{pos}-{pos}/{alt}"
    missense, synonymous, splice = 0, 0, 0

    try:
        response = requests.get(server + endpoint, headers={"Content-Type": "application/json"})
        if response.ok:
            data = response.json()[0]
            for t in data.get('transcript_consequences', []):
                terms = t.get('consequence_terms', [])

                # The "Wrong Ingredient"
                if 'missense_variant' in terms: missense = 1
                # The "Harmless Swap"
                if 'synonymous_variant' in terms: synonymous = 1
                # The "Broken Scissors"
                if any('splice' in term for term in terms): splice = 1
    except Exception:
        pass

    is_missense_list.append(missense)
    is_synonymous_list.append(synonymous)
    is_splice_list.append(splice)

    time.sleep(0.1)

df_test['Is_Missense'] = is_missense_list
df_test['Is_Synonymous'] = is_synonymous_list
df_test['Is_Splice_Site'] = is_splice_list

print("\nResults for the first 50 rows:")
print(df_test[['Variant_ID', 'Is_Missense', 'Is_Synonymous', 'Is_Splice_Site']].head(10))

# Save the upgraded dataset!
output_name = 'enriched_training_data.csv'
df.to_csv(output_name, index=False)
print(f"✅ Success! Your new dataset is saved as {output_name}")

In [ ]:
df_test.to_csv('enriched_training_data.csv', index=False)

In [1]:
#GENERATING ADA BOOST FROM DATASET-V1

!pip install pandas scikit-learn
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib

print("--- Phase 1: Loading the Genetic Dataset ---")
# 1. Load your perfectly formatted data
try:
    df = pd.read_csv('/kaggle/input/datasets/rohanalexbimal/enriched-training-data/enriched_training_data.csv')
except FileNotFoundError:
    print("Error: Could not find 'final_training_data.csv'. Make sure it's in the same folder!")
    exit()

# Drop any accidental blank rows
df = df.dropna()
print(f"Total variants ready for training: {len(df)}")

# 2. Separate the Clues (X) from the Answer (y)
# We drop Variant_ID because the AI can't do math on a text string like "17:41244408:T:C"
feature_columns = ['gnomAD_AF', 'SIFT_Score', 'CADD_Score', 'Is_Nonsense', 'Is_Missense','Is_Synonymous','Is_Splice_Site']
X = df[feature_columns]
y = df['Is_Pathogenic']

# 3. Split the data: 80% for studying (training), 20% for the final exam (testing)
# random_state=42 ensures you get the exact same split every time you run this script
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("\n--- Phase 2: Training the Random Forest AI ---")
# Initialize the model with 100 decision trees
# 'class_weight=balanced' tells the AI to pay equal attention to Benign and Pathogenic rows
model = AdaBoostClassifier(n_estimators=100, random_state=42)

# This is where the actual mathematical learning happens!
model.fit(X_train, y_train)
print("Training complete! 🚀")

print("\n--- Phase 3: Testing AI Accuracy ---")
# 4. Give the AI its final exam on the 20% of data it hasn't seen yet
y_pred = model.predict(X_test)

# Print out the official scorecard
accuracy = accuracy_score(y_test, y_pred)
print(f"Overall Accuracy: {accuracy * 100:.2f}%\n")

print("Detailed Clinical Report:")
# This breaks down False Positives and False Negatives
print(classification_report(y_test, y_pred, target_names=['Benign (0)', 'Pathogenic (1)']))

print("\n--- Phase 4: Feature Importance ---")
# 5. Find out exactly which clues the AI relied on the most to make its decisions
importances = model.feature_importances_
for feature, importance in zip(feature_columns, importances):
    print(f" - {feature}: {importance * 100:.1f}%")

# 6. Save the AI's brain to a file so your web app frontend can use it later
model_filename = "synaptix_clinical_ai.pkl"
joblib.dump(model, model_filename)
print(f"\n✅ SUCCESS: AI Model saved to your folder as '{model_filename}'")

--- Phase 1: Loading the Genetic Dataset ---
Total variants ready for training: 12000

--- Phase 2: Training the Random Forest AI ---
Training complete! 🚀

--- Phase 3: Testing AI Accuracy ---
Overall Accuracy: 97.62%

Detailed Clinical Report:
                precision    recall  f1-score   support

    Benign (0)       0.98      0.97      0.98      1210
Pathogenic (1)       0.97      0.98      0.98      1190

      accuracy                           0.98      2400
     macro avg       0.98      0.98      0.98      2400
  weighted avg       0.98      0.98      0.98      2400


--- Phase 4: Feature Importance ---
 - gnomAD_AF: 38.5%
 - SIFT_Score: 1.4%
 - CADD_Score: 44.2%
 - Is_Nonsense: 7.7%
 - Is_Missense: 1.5%
 - Is_Synonymous: 2.0%
 - Is_Splice_Site: 4.6%

✅ SUCCESS: AI Model saved to your folder as 'synaptix_clinical_ai.pkl'


In [2]:
#GENERATING RANDOM FOREST FROM DATASET-V1

!pip install pandas scikit-learn
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib

print("--- Phase 1: Loading the Genetic Dataset ---")
# 1. Load your perfectly formatted data
try:
    df = pd.read_csv('/kaggle/input/datasets/rohanalexbimal/enriched-training-data/enriched_training_data.csv')
except FileNotFoundError:
    print("Error: Could not find 'final_training_data.csv'. Make sure it's in the same folder!")
    exit()

# Drop any accidental blank rows
df = df.dropna()
print(f"Total variants ready for training: {len(df)}")

# 2. Separate the Clues (X) from the Answer (y)
# We drop Variant_ID because the AI can't do math on a text string like "17:41244408:T:C"
feature_columns = ['gnomAD_AF', 'SIFT_Score', 'CADD_Score', 'Is_Nonsense', 'Is_Missense','Is_Synonymous','Is_Splice_Site']
X = df[feature_columns]
y = df['Is_Pathogenic']

# 3. Split the data: 80% for studying (training), 20% for the final exam (testing)
# random_state=42 ensures you get the exact same split every time you run this script
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("\n--- Phase 2: Training the Random Forest AI ---")
# Initialize the model with 100 decision trees
# 'class_weight=balanced' tells the AI to pay equal attention to Benign and Pathogenic rows
model = RandomForestClassifier(n_estimators=100, random_state=42)

# This is where the actual mathematical learning happens!
model.fit(X_train, y_train)
print("Training complete! 🚀")

print("\n--- Phase 3: Testing AI Accuracy ---")
# 4. Give the AI its final exam on the 20% of data it hasn't seen yet
y_pred = model.predict(X_test)

# Print out the official scorecard
accuracy = accuracy_score(y_test, y_pred)
print(f"Overall Accuracy: {accuracy * 100:.2f}%\n")

print("Detailed Clinical Report:")
# This breaks down False Positives and False Negatives
print(classification_report(y_test, y_pred, target_names=['Benign (0)', 'Pathogenic (1)']))

print("\n--- Phase 4: Feature Importance ---")
# 5. Find out exactly which clues the AI relied on the most to make its decisions
importances = model.feature_importances_
for feature, importance in zip(feature_columns, importances):
    print(f" - {feature}: {importance * 100:.1f}%")

# 6. Save the AI's brain to a file so your web app frontend can use it later
model_filename = "synaptix_clinical_ai_ada.pkl"
joblib.dump(model, model_filename)
print(f"\n✅ SUCCESS: AI Model saved to your folder as '{model_filename}'")

--- Phase 1: Loading the Genetic Dataset ---
Total variants ready for training: 12000

--- Phase 2: Training the Random Forest AI ---
Training complete! 🚀

--- Phase 3: Testing AI Accuracy ---
Overall Accuracy: 97.67%

Detailed Clinical Report:
                precision    recall  f1-score   support

    Benign (0)       0.98      0.98      0.98      1210
Pathogenic (1)       0.98      0.98      0.98      1190

      accuracy                           0.98      2400
     macro avg       0.98      0.98      0.98      2400
  weighted avg       0.98      0.98      0.98      2400


--- Phase 4: Feature Importance ---
 - gnomAD_AF: 38.2%
 - SIFT_Score: 3.9%
 - CADD_Score: 45.2%
 - Is_Nonsense: 7.8%
 - Is_Missense: 0.7%
 - Is_Synonymous: 2.8%
 - Is_Splice_Site: 1.4%

✅ SUCCESS: AI Model saved to your folder as 'synaptix_clinical_ai_ada.pkl'
